# PyTorch 第十章：大语言模型安装与应用

> 对应《PyTorch 实用教程（第二版）》第十章  
> 目标：不下载大型权重，用最小可运行实验理解 **LLM 本地部署、聊天模板、生成循环、流式输出、KV Cache、量化、上下文管理、工具角色与 LLM 应用工程**。

## 原教程小节顺序

1. 10.1 Qwen 部署与分析
2. 10.2 ChatGLM3 部署与分析
3. 10.3 Baichuan2 部署与分析
4. 10.4 Yi 部署与分析
5. 10.5 GPT Academic 安装与使用

教程章节入口：  
https://tingsongyu.github.io/PyTorch-Tutorial-2nd/chapter-10/

## 本 Notebook 的原则

原教程写于 2024 年，使用的具体模型版本与安装命令已经具有明显时代性。本 Notebook：

- **保持原教程小节顺序和核心知识点**；
- 不下载 Qwen / ChatGLM / Baichuan / Yi 的 GB 级权重；
- 用小型 PyTorch 模型和模拟数据复现共性机制；
- 将过时的模型加载、聊天模板、量化和缓存概念更新为现代 Hugging Face / PyTorch 思路；
- 重点服务后续 LLM / Agent 学习，而不是记忆旧版仓库命令。

## 0. 环境导入

In [ ]:
import math
import random
from collections import Counter
from dataclasses import dataclass

import numpy as np
import torch
import torch.nn as nn
import torch.nn.functional as F

SEED = 42
random.seed(SEED)
np.random.seed(SEED)
torch.manual_seed(SEED)

device = torch.device("cuda" if torch.cuda.is_available() else "cpu")

print("PyTorch:", torch.__version__)
print("device:", device)

## 第十章先抓住一个统一框架

Qwen、ChatGLM3、Baichuan2、Yi 虽然仓库、特殊 token、模型层细节不同，但聊天推理主流程高度一致：

```text
messages
  ↓
chat template
  ↓
tokenizer
  ↓
input_ids / attention_mask
  ↓
Causal LLM
  ↓
last-token logits
  ↓
sampling / decoding
  ↓
next token
  ↓
KV Cache + 自回归循环
  ↓
EOS / stopping criteria
```

因此，本章真正值得掌握的是**共性推理框架**，而不是背某个 2024 版模型的安装命令。

## 当前生态与原教程的关键差异

现代 Transformers 中更常见的是：

```python
tokenizer = AutoTokenizer.from_pretrained(model_id)
model = AutoModelForCausalLM.from_pretrained(
    model_id,
    device_map="auto",
    dtype="auto",
)

inputs = tokenizer.apply_chat_template(
    messages,
    add_generation_prompt=True,
    tokenize=True,
    return_dict=True,
    return_tensors="pt",
)

outputs = model.generate(**inputs, max_new_tokens=128)
```

注意：这里只作为**现代接口参考**放在 Markdown 中，不在本 Notebook 下载真实模型。

另外，原教程中“Transformers 没有 KV Cache”的描述已过时。现代生成模型普遍使用 KV Cache 避免每个解码步重复计算历史 token 的 K/V。

## 0.1 一个贯穿本章的极简 Tokenizer

真实 LLM 使用 BPE / SentencePiece / byte-level tokenizer。这里仅用字符级 tokenizer，让后续生成实验完全离线可运行。

In [ ]:
class TinyTokenizer:
    def __init__(self, texts):
        chars = sorted(set("".join(texts)))
        special = ["<pad>", "<bos>", "<eos>", "<unk>"]
        self.id_to_token = special + chars
        self.token_to_id = {t: i for i, t in enumerate(self.id_to_token)}
        self.pad_token_id = self.token_to_id["<pad>"]
        self.bos_token_id = self.token_to_id["<bos>"]
        self.eos_token_id = self.token_to_id["<eos>"]
        self.unk_token_id = self.token_to_id["<unk>"]

    def encode(self, text, add_bos=False):
        ids = [self.token_to_id.get(ch, self.unk_token_id) for ch in text]
        if add_bos:
            ids = [self.bos_token_id] + ids
        return ids

    def decode(self, ids):
        tokens = []
        for i in ids:
            token = self.id_to_token[int(i)]
            if token in {"<pad>", "<bos>", "<eos>"}:
                continue
            tokens.append(token)
        return "".join(tokens)

corpus = [
    "你好，我是一个小模型。",
    "用户提出问题，助手生成回答。",
    "大语言模型使用自回归生成。",
]
tiny_tokenizer = TinyTokenizer(corpus)

ids = tiny_tokenizer.encode("你好", add_bos=True)
print("ids:", ids)
print("decode:", tiny_tokenizer.decode(ids))
print("vocab size:", len(tiny_tokenizer.id_to_token))

## 0.2 一个极简 Decoder-only Causal LM

它不是任何真实 Qwen / ChatGLM / Yi 模型，只承担两个作用：

1. 展示统一的 `input_ids → logits` 接口；
2. 用于后面演示自回归、采样和流式输出。

In [ ]:
class TinyCausalLM(nn.Module):
    def __init__(self, vocab_size, d_model=32, nhead=4, num_layers=2, max_len=256):
        super().__init__()
        self.token_emb = nn.Embedding(vocab_size, d_model)
        self.pos_emb = nn.Embedding(max_len, d_model)

        layer = nn.TransformerEncoderLayer(
            d_model=d_model,
            nhead=nhead,
            dim_feedforward=64,
            dropout=0.0,
            batch_first=True,
        )
        self.blocks = nn.TransformerEncoder(layer, num_layers=num_layers)
        self.norm = nn.LayerNorm(d_model)
        self.lm_head = nn.Linear(d_model, vocab_size, bias=False)

    def forward(self, input_ids):
        B, T = input_ids.shape
        pos = torch.arange(T, device=input_ids.device).unsqueeze(0)
        x = self.token_emb(input_ids) + self.pos_emb(pos)

        causal_mask = torch.triu(
            torch.ones(T, T, dtype=torch.bool, device=input_ids.device),
            diagonal=1,
        )
        h = self.blocks(x, mask=causal_mask)
        h = self.norm(h)
        return self.lm_head(h)

tiny_lm = TinyCausalLM(len(tiny_tokenizer.id_to_token)).to(device)

test_ids = torch.tensor(
    [tiny_tokenizer.encode("你好", add_bos=True)],
    device=device,
)
logits = tiny_lm(test_ids)

print("input:", test_ids.shape)
print("logits:", logits.shape)
print("last-token logits:", logits[:, -1].shape)

# 10.1 Qwen 部署与分析

原教程本节重点：

1. Qwen 本地安装与模型权重选择
2. Hugging Face `AutoModelForCausalLM` 模型结构
3. 流式推理流程
4. 多轮对话 Prompt 拼装
5. 显存与上下文长度关系

原教程分析的是 Qwen / Qwen1.5 时代模型。今天更应学习这些**模型无关机制**。

## 模型代码结构：为什么各种 LLM 都能用 AutoModel？

典型结构可以抽象成：

```text
AutoModelForCausalLM
        ↓
具体模型 ForCausalLM
        ↓
Base Transformer Model
        ↓
Repeated Decoder Blocks
        ↓
Attention + MLP + Norm
        ↓
lm_head
```

`lm_head` 把最后的 hidden state 投影到词表维度：

`[B,T,H] → [B,T,V]`

In [ ]:
B, T, H, V = 2, 5, 16, 100
hidden = torch.randn(B, T, H)
lm_head = nn.Linear(H, V, bias=False)
logits = lm_head(hidden)

print("hidden:", hidden.shape)
print("vocab logits:", logits.shape)

## 多轮对话：LLM 本身通常不“记得上一轮”

应用层保存：

```python
messages = [
    {"role": "system", "content": "..."},
    {"role": "user", "content": "..."},
    {"role": "assistant", "content": "..."},
    {"role": "user", "content": "..."},
]
```

下一轮再把这些历史消息重新格式化后输入模型。

所谓“对话记忆”，首先是**上下文管理问题**。

In [ ]:
messages = [
    {"role": "system", "content": "你是一个简洁的助手。"},
    {"role": "user", "content": "我叫小明。"},
    {"role": "assistant", "content": "你好，小明。"},
    {"role": "user", "content": "我叫什么？"},
]

for m in messages:
    print(f"{m['role']:>9}: {m['content']}")

## Chat Template：不要手写死每个模型的特殊 token

Qwen 经典 ChatML 风格使用类似：

`<|im_start|>role ... <|im_end|>`

但不同模型的特殊 token 可能完全不同。

现代 Transformers 的正确思路是让 tokenizer 自己通过 `apply_chat_template()` 按模型训练时的格式组织消息。

In [ ]:
def qwen_style_template(messages, add_generation_prompt=True):
    parts = []
    for m in messages:
        parts.append(
            f"<|im_start|>{m['role']}\n"
            f"{m['content']}<|im_end|>\n"
        )
    if add_generation_prompt:
        parts.append("<|im_start|>assistant\n")
    return "".join(parts)

formatted = qwen_style_template(messages)
print(formatted)

### 为什么 Chat Template 很重要？

Chat 模型本质仍在做“续写”。

控制 token 告诉模型：

- 哪部分是 system；
- 哪部分是 user；
- 哪部分是 assistant；
- 现在应该开始生成 assistant 内容。

格式与训练分布不一致时，模型性能会明显下降。

## 自回归生成：一次 forward 得到所有位置 logits，但只取最后位置预测下一个 token

In [ ]:
@torch.inference_mode()
def greedy_next_token(model, input_ids):
    logits = model(input_ids)
    next_token_logits = logits[:, -1, :]
    return next_token_logits.argmax(dim=-1, keepdim=True)

prompt = torch.tensor(
    [tiny_tokenizer.encode("你好", add_bos=True)],
    device=device,
)

next_id = greedy_next_token(tiny_lm, prompt)
print("next token id:", next_id.item())
print("next token:", tiny_tokenizer.decode(next_id[0].cpu().tolist()))

## 一个最小生成循环

停止条件通常包括：

- EOS token
- `max_new_tokens`
- 自定义 stopping criteria

In [ ]:
@torch.inference_mode()
def greedy_generate(model, input_ids, eos_token_id=None, max_new_tokens=8):
    generated = input_ids.clone()

    for _ in range(max_new_tokens):
        next_token = greedy_next_token(model, generated)
        generated = torch.cat([generated, next_token], dim=1)

        if eos_token_id is not None and torch.all(next_token == eos_token_id):
            break

    return generated

generated = greedy_generate(
    tiny_lm,
    prompt,
    eos_token_id=tiny_tokenizer.eos_token_id,
    max_new_tokens=5,
)

print("generated shape:", generated.shape)
print("generated ids:", generated[0].cpu().tolist())

## 流式输出（streaming）

流式返回并不改变模型本身的生成机制。

本质仍是：

1. 生成一个 token；
2. decode；
3. `yield` 给 UI / 网络连接；
4. 继续生成下一个 token。

In [ ]:
def fake_stream(token_ids, tokenizer):
    current = []
    for tid in token_ids:
        current.append(tid)
        yield tokenizer.decode(current)

demo_ids = tiny_tokenizer.encode("你好，我是")
for partial_text in fake_stream(demo_ids, tiny_tokenizer):
    print(partial_text)

## KV Cache：本章最重要的部署概念之一

Attention 中历史 token 的 K/V 在后续生成步骤不会改变。

没有 cache 时，第 $t$ 步可能重新计算前面全部 token。

有 KV Cache 时：

- 历史 K/V 保留；
- 新一步只计算新 token 的 Q/K/V；
- 新 Q 与缓存中的历史 K 做 attention。

因此 KV Cache 用**显存换速度**。

In [ ]:
# 只演示缓存张量如何随 token 增长，不实现完整高性能 attention
B = 1
num_kv_heads = 2
head_dim = 8

k_cache = torch.empty(B, num_kv_heads, 0, head_dim)
v_cache = torch.empty(B, num_kv_heads, 0, head_dim)

for step in range(4):
    new_k = torch.randn(B, num_kv_heads, 1, head_dim)
    new_v = torch.randn(B, num_kv_heads, 1, head_dim)

    k_cache = torch.cat([k_cache, new_k], dim=2)
    v_cache = torch.cat([v_cache, new_v], dim=2)

    print(
        f"step={step+1}",
        "K:", tuple(k_cache.shape),
        "V:", tuple(v_cache.shape),
    )

## KV Cache 显存估算

忽略额外开销，可粗略估计：

$$
M_{KV}
\approx
2 \times L \times B \times S
\times H_{kv} \times D_h
\times \text{bytes}
$$

其中：

- 2：K 和 V
- $L$：层数
- $B$：batch size
- $S$：缓存序列长度
- $H_{kv}$：KV heads
- $D_h$：head dimension

因此上下文越长，KV Cache 通常近似**线性增长**。

注意：模型的实际总显存还包含权重、临时激活、allocator 缓存、kernel workspace 等，不能只靠这一条公式预测峰值。

In [ ]:
def kv_cache_gib(
    layers,
    batch,
    seq_len,
    num_kv_heads,
    head_dim,
    bytes_per_elem=2,
):
    total_bytes = (
        2 * layers * batch * seq_len
        * num_kv_heads * head_dim
        * bytes_per_elem
    )
    return total_bytes / (1024 ** 3)

for seq in [1024, 4096, 16384]:
    gib = kv_cache_gib(
        layers=32,
        batch=1,
        seq_len=seq,
        num_kv_heads=8,
        head_dim=128,
        bytes_per_elem=2,
    )
    print(f"{seq:>5} tokens -> {gib:.3f} GiB")

# 10.2 ChatGLM3 部署与分析

原教程本节重点：

1. ChatGLM3 本地部署与 INT4
2. 模型类层级
3. `stream_generate()` 循环
4. Prompt 角色设计
5. `observation` 角色与工具调用
6. 显存 / 上下文实验

与 10.1 重复的部署细节这里不重复，重点看 **生成循环与工具闭环**。

## 流式生成循环的四步

教程从 ChatGLM3 源码中提炼出：

1. `prepare_inputs_for_generation`
2. forward 得到最后位置 logits
3. logits processor / sampling
4. yield + stopping condition

这是 Hugging Face `GenerationMixin.generate()` 背后的通用思路。

In [ ]:
def temperature_sample(logits, temperature=1.0):
    if temperature <= 0:
        raise ValueError("temperature must be > 0")
    probs = F.softmax(logits / temperature, dim=-1)
    return torch.multinomial(probs, num_samples=1)

demo_logits = torch.tensor([2.0, 1.0, 0.5])

for temp in [0.5, 1.0, 2.0]:
    probs = F.softmax(demo_logits / temp, dim=-1)
    print(f"T={temp}:", [round(float(x), 3) for x in probs])

## `observation`：为什么对 Agent 很重要？

教程中的 ChatGLM3 支持：

- system
- user
- assistant
- observation

其中 observation 表示**外部工具执行结果**。

Agent 闭环不是：

`LLM → tool call → LLM`

而是：

`LLM → tool call → execute tool → observation → LLM`

如果工具结果没有回写，模型就无法根据真实执行结果继续决策。

In [ ]:
agent_messages = [
    {"role": "system", "content": "必要时使用计算器。"},
    {"role": "user", "content": "23*17 等于多少？"},
    {"role": "assistant", "content": "调用 calculator(a=23, b=17)"},
    {"role": "observation", "content": "391"},
    {"role": "assistant", "content": "23*17=391。"},
]

for msg in agent_messages:
    print(f"[{msg['role']}] {msg['content']}")

## 一个极小工具闭环

这里不用 LLM，只演示控制流。

In [ ]:
TOOLS = {
    "multiply": lambda a, b: a * b,
}

tool_call = {
    "name": "multiply",
    "arguments": {"a": 23, "b": 17},
}

result = TOOLS[tool_call["name"]](**tool_call["arguments"])

observation = {
    "role": "observation",
    "content": str(result),
}

print(observation)

### 常见错误

- 只保存 tool call，不保存 tool result；
- 把失败信息丢弃；
- 工具参数没有结构化校验；
- 无限调用工具，没有 `max_steps`；
- 把工具输出直接当可信指令，而不是不可信外部数据。

这些问题比“模型是哪一代 ChatGLM”更值得长期掌握。

# 10.3 Baichuan2 部署与分析

原教程重点：

1. Baichuan2 7B / 13B 与量化版本
2. `AutoModelForCausalLM` / tokenizer / generation config
3. 三角色 Prompt
4. 特殊 role token
5. 上下文长度与左截断
6. 显存变化

本节最值得保留的是：**上下文预算与量化**。

## Context Window 预算

若：

- 最大上下文长度 = `max_context`
- 最多还要生成 = `max_new_tokens`

则 prompt 最多可占：

`max_context - max_new_tokens`

超过时需要截断、摘要或外部记忆。

In [ ]:
def truncate_left(input_ids, max_context, max_new_tokens):
    max_input = max_context - max_new_tokens
    if max_input <= 0:
        raise ValueError("max_new_tokens must be smaller than max_context")
    return input_ids[-max_input:]

tokens = list(range(20))

kept = truncate_left(
    tokens,
    max_context=12,
    max_new_tokens=4,
)

print("original:", tokens)
print("kept:", kept)
print("input budget:", 12 - 4)

## 为什么简单“左截断”有风险？

它可能删除：

- system instruction
- 用户最早定义的约束
- 关键事实
- 工具返回结果

更成熟的上下文管理会组合：

- 固定保留 system
- 最近窗口
- 历史摘要
- RAG / external memory
- 按 token budget 动态裁剪

In [ ]:
def keep_system_and_recent(messages, max_chars=60):
    system = [m for m in messages if m["role"] == "system"]
    others = [m for m in messages if m["role"] != "system"]

    kept = []
    used = 0
    for m in reversed(others):
        size = len(m["content"])
        if used + size > max_chars:
            break
        kept.append(m)
        used += size

    return system + list(reversed(kept))

long_history = [
    {"role": "system", "content": "始终用中文回答。"},
    {"role": "user", "content": "第一轮：介绍模型。"},
    {"role": "assistant", "content": "这是第一轮回答。"},
    {"role": "user", "content": "第二轮：继续解释。"},
    {"role": "assistant", "content": "这是第二轮回答。"},
    {"role": "user", "content": "第三轮：总结。"},
]

trimmed = keep_system_and_recent(long_history, max_chars=35)
for m in trimmed:
    print(m)

## 参数显存：先做数量级估算

只考虑权重：

$$
M_{weights}pprox N_{params}	imes 	ext{bytes per parameter}
$$

例如理论上：

- FP32：4 bytes / param
- FP16 / BF16：2 bytes / param
- INT8：约 1 byte / param
- INT4：约 0.5 byte / param

实际量化模型还存在 scale、zero-point、group metadata、未量化层等额外开销。

In [ ]:
def weight_memory_gib(num_params_billion, bits_per_param):
    total_bits = num_params_billion * 1e9 * bits_per_param
    return total_bits / 8 / (1024 ** 3)

for bits in [32, 16, 8, 4]:
    print(
        f"7B @ {bits:>2}-bit:",
        f"{weight_memory_gib(7, bits):.2f} GiB theoretical weights"
    )

## 最小 Fake Quantization：理解“低比特权重”在做什么

真实 AWQ / GPTQ / bitsandbytes 远比下面复杂。这里仅演示：

1. 浮点权重映射到有限整数区间；
2. 保存量化整数 + scale；
3. 推理时恢复近似浮点值。

量化引入误差，换来更低的存储与显存压力。

In [ ]:
def symmetric_fake_quantize(x, bits=4):
    qmax = 2 ** (bits - 1) - 1
    scale = x.abs().max() / max(qmax, 1)

    if scale == 0:
        return x.clone(), torch.zeros_like(x, dtype=torch.int8), scale

    q = torch.round(x / scale).clamp(-qmax, qmax).to(torch.int8)
    x_hat = q.float() * scale
    return x_hat, q, scale

w = torch.randn(128, 128)
w_hat, q, scale = symmetric_fake_quantize(w, bits=4)

mae = (w - w_hat).abs().mean()

print("float weight dtype:", w.dtype)
print("quantized storage dtype in demo:", q.dtype)
print("scale:", round(float(scale), 6))
print("mean abs reconstruction error:", round(float(mae), 6))

### 当前 Hugging Face 量化思路

真实项目中常见现代写法类似：

```python
from transformers import AutoModelForCausalLM, BitsAndBytesConfig
import torch

quant_config = BitsAndBytesConfig(
    load_in_4bit=True,
    bnb_4bit_quant_type="nf4",
    bnb_4bit_compute_dtype=torch.bfloat16,
)

model = AutoModelForCausalLM.from_pretrained(
    model_id,
    device_map="auto",
    dtype="auto",
    quantization_config=quant_config,
)
```

不要照抄原教程中的旧版 Windows wheel、固定 `bitsandbytes==0.41.1` 等环境方案；量化库与 CUDA / PyTorch 兼容性变化很快，应以当前官方文档为准。

# 10.4 Yi 部署与分析

原教程本节重点：

1. Yi 模型安装与 AWQ / GPTQ 量化
2. `LlamaForCausalLM` 类结构
3. LlamaDecoderLayer
4. Q/K/V 投影
5. RoPE
6. RMSNorm
7. MLP
8. GQA / KV heads
9. `apply_chat_template`
10. `TextIteratorStreamer`

这一节对后续理解现代 LLM 架构非常重要。

## Yi 展示出的现代 Decoder Block 骨架

教程分析的 Yi 权重配置可抽象为：

```text
LlamaForCausalLM
└── LlamaModel
    ├── Embedding
    ├── N × DecoderLayer
    │   ├── RMSNorm
    │   ├── Self-Attention
    │   │   ├── q_proj
    │   │   ├── k_proj
    │   │   ├── v_proj
    │   │   ├── RoPE
    │   │   └── o_proj
    │   ├── residual
    │   ├── RMSNorm
    │   ├── gated MLP
    │   └── residual
    ├── RMSNorm
    └── LM Head
```

这已经非常接近今天许多 decoder-only LLM 的基本骨架。

## RMSNorm

与 LayerNorm 不同，RMSNorm 不做均值中心化，核心形式可写为：

$$
\mathrm{RMSNorm}(x)
=
\frac{x}{\sqrt{\frac{1}{d}\sum_i x_i^2+\epsilon}}
\odot w
$$

In [ ]:
class RMSNorm(nn.Module):
    def __init__(self, dim, eps=1e-6):
        super().__init__()
        self.weight = nn.Parameter(torch.ones(dim))
        self.eps = eps

    def forward(self, x):
        rms = x.pow(2).mean(dim=-1, keepdim=True)
        x_norm = x * torch.rsqrt(rms + self.eps)
        return self.weight * x_norm

x = torch.randn(2, 4, 8)
rms_norm = RMSNorm(8)
y = rms_norm(x)

print("shape:", y.shape)
print("mean RMS after norm:", round(float(y.pow(2).mean().sqrt()), 4))

## RoPE：把位置信息旋转进 Q / K

Rotary Positional Embedding（RoPE）的关键思想不是把一个位置向量直接加到 hidden state，而是对 Q/K 的二维子空间做与位置相关的旋转。

二维旋转：

$$
\begin{bmatrix}
x'_1\\
x'_2
\end{bmatrix}
=
\begin{bmatrix}
\cos\theta & -\sin\theta\\
\sin\theta & \cos\theta
\end{bmatrix}
\begin{bmatrix}
x_1\\
x_2
\end{bmatrix}
$$

模型通过相对旋转角度获得相对位置信息。

In [ ]:
def rotate_pair(x1, x2, theta):
    cos_t = torch.cos(theta)
    sin_t = torch.sin(theta)
    y1 = x1 * cos_t - x2 * sin_t
    y2 = x1 * sin_t + x2 * cos_t
    return y1, y2

x1 = torch.tensor(1.0)
x2 = torch.tensor(0.0)

for pos in [0, 1, 2]:
    theta = torch.tensor(pos * 0.5)
    y1, y2 = rotate_pair(x1, x2, theta)
    print(pos, round(float(y1), 3), round(float(y2), 3))

## MHA、MQA、GQA：为什么 KV heads 可以比 Q heads 少？

- MHA：Q/K/V 都有相同数量的 heads
- MQA：所有 query heads 共享一组 K/V
- GQA：多个 query heads 分组共享 K/V

GQA 在保持较好模型能力的同时，显著减少 KV Cache。

In [ ]:
num_query_heads = 32
head_dim = 128

configs = {
    "MHA": 32,
    "GQA": 8,
    "MQA": 1,
}

for name, kv_heads in configs.items():
    mem = kv_cache_gib(
        layers=32,
        batch=1,
        seq_len=8192,
        num_kv_heads=kv_heads,
        head_dim=head_dim,
        bytes_per_elem=2,
    )
    print(f"{name}: kv_heads={kv_heads:>2}, KV cache={mem:.3f} GiB")

## Gated MLP：SiLU / SwiGLU 一类结构

现代 LLM 的 FFN 往往不再是简单：

`Linear → ReLU → Linear`

而是 gated 结构，例如：

`down( SiLU(gate(x)) * up(x) )`

In [ ]:
class GatedMLP(nn.Module):
    def __init__(self, d_model, hidden_dim):
        super().__init__()
        self.gate_proj = nn.Linear(d_model, hidden_dim, bias=False)
        self.up_proj = nn.Linear(d_model, hidden_dim, bias=False)
        self.down_proj = nn.Linear(hidden_dim, d_model, bias=False)

    def forward(self, x):
        return self.down_proj(
            F.silu(self.gate_proj(x)) * self.up_proj(x)
        )

mlp = GatedMLP(16, 32)
x = torch.randn(2, 5, 16)
print("input:", x.shape)
print("output:", mlp(x).shape)

## Chat Template + Streamer

原教程 Yi 部分已经使用现代化程度较高的思路：

1. 历史消息转成 `messages`
2. `tokenizer.apply_chat_template(...)`
3. 构造 generation kwargs
4. `model.generate(...)`
5. `TextIteratorStreamer` 异步读取输出

当前仍应保留这个架构，但具体参数以当前模型仓库 / Transformers 文档为准。

# 10.5 GPT Academic 安装与使用

原教程没有继续分析另一个基础模型，而是转向一个真实 LLM 应用。

它最值得学习的不是某个按钮，而是：

1. Prompt 模板化
2. 批量任务自动化
3. 长文档分块
4. Map-Reduce 式汇总
5. 代码项目逐文件分析
6. RAG / 知识库问答
7. 插件化工程结构

这就是“模型能力”变成“生产力工具”的过程。

## 基础功能 = Prompt Template + 用户输入

例如“学术润色”“语法检查”“中英翻译”，本质都是：

```text
system / instruction template
        +
user content
        ↓
LLM
```

重要的不是按钮名称，而是把重复任务固化为可靠的模板。

In [ ]:
PROMPT_TEMPLATES = {
    "summarize": "请用三句话概括下面内容：\n{text}",
    "explain_code": "请解释以下代码的功能、关键数据流和风险：\n```python\n{text}\n```",
}

user_text = "Transformer 使用注意力机制建模 token 之间的依赖关系。"
prompt = PROMPT_TEMPLATES["summarize"].format(text=user_text)

print(prompt)

## 长文档：为什么要 chunk？

如果文档超过模型上下文长度，不能把全文硬塞进去。

常见策略：

1. 文档切块；
2. 每块单独处理；
3. 汇总块级结果；
4. 必要时再次递归汇总。

这就是常见的 Map-Reduce summarization。

In [ ]:
def chunk_text(text, chunk_size=40):
    return [
        text[i:i + chunk_size]
        for i in range(0, len(text), chunk_size)
    ]

long_text = (
    "第一段介绍PyTorch。第二段介绍Transformer。"
    "第三段介绍大语言模型。第四段介绍RAG。"
    "第五段介绍Agent与工具调用。"
)

chunks = chunk_text(long_text, chunk_size=24)

for i, chunk in enumerate(chunks, 1):
    print(i, repr(chunk))

## 最小 Map-Reduce 流程

这里不用真正 LLM，用确定性函数模拟“每块摘要 → 总摘要”的控制流。

In [ ]:
def mock_summarize(text, max_chars=16):
    return text[:max_chars] + ("..." if len(text) > max_chars else "")

partial_summaries = [mock_summarize(c) for c in chunks]
final_summary = " | ".join(partial_summaries)

print("map:")
for s in partial_summaries:
    print("-", s)

print("\nreduce:")
print(final_summary)

## 代码项目分析：不要直接把整个仓库塞给模型

教程中 GPT Academic 的思路是：

1. 遍历项目文件；
2. 每个文件独立摘要；
3. 汇总文件级摘要；
4. 最后生成项目级说明。

这比“把整个 ZIP 解压后全部粘进 prompt”更可控，也更容易定位错误。

In [ ]:
fake_project = {
    "model.py": "class Model: pass",
    "train.py": "def train(): pass",
    "data.py": "def load_data(): pass",
}

file_summaries = {}

for filename, source in fake_project.items():
    first_line = source.splitlines()[0]
    file_summaries[filename] = f"{filename}: {first_line}"

for summary in file_summaries.values():
    print(summary)

## RAG：知识库问答不等于“把 PDF 全文拼进 Prompt”

典型流程：

```text
documents
   ↓
chunk
   ↓
embedding
   ↓
vector index
   ↓
query embedding
   ↓
top-k retrieval
   ↓
retrieved context + question
   ↓
LLM
```

下面用词袋向量模拟 retrieval，不需要任何 embedding 模型。

In [ ]:
docs = [
    "PyTorch autograd 用于自动求导",
    "Transformer 通过 attention 建模序列",
    "RAG 先检索相关文档再让语言模型生成",
    "量化可以降低大模型权重显存",
]

def tokenize_zh_simple(text):
    # 为了最小演示，直接使用字符级 token。
    return list(text.replace(" ", ""))

vocab_rag = sorted(set(
    token
    for doc in docs
    for token in tokenize_zh_simple(doc)
))
token_to_col = {t: i for i, t in enumerate(vocab_rag)}

def bow_vector(text):
    v = torch.zeros(len(vocab_rag))
    for token in tokenize_zh_simple(text):
        if token in token_to_col:
            v[token_to_col[token]] += 1
    return F.normalize(v.unsqueeze(0), dim=1).squeeze(0)

doc_matrix = torch.stack([bow_vector(d) for d in docs])

query = "如何减少模型显存"
q = bow_vector(query)

scores = doc_matrix @ q
top = scores.topk(k=2)

for score, idx in zip(top.values, top.indices):
    print(round(float(score), 3), docs[int(idx)])

## 插件化：LLM 应用应该把“模型调用”和“业务工具”分离

一个更健康的工程结构：

```text
app/
├── llm_backend.py       # 模型/API适配
├── prompts.py           # Prompt模板
├── chunking.py          # 文档切分
├── retrieval.py         # 检索
├── tools/               # 工具插件
├── workflows/           # 批处理/论文/代码分析流程
└── web.py               # UI/API
```

好处：

- 能单独替换模型；
- 能单测每个工具；
- Prompt 与业务逻辑不混在 UI 里；
- RAG / Agent 更容易演进。

In [ ]:
class ToolRegistry:
    def __init__(self):
        self.tools = {}

    def register(self, name, func):
        self.tools[name] = func

    def call(self, name, **kwargs):
        if name not in self.tools:
            raise KeyError(f"unknown tool: {name}")
        return self.tools[name](**kwargs)

registry = ToolRegistry()
registry.register("word_count", lambda text: len(text.split()))
registry.register("char_count", lambda text: len(text))

print(registry.call("char_count", text="PyTorch LLM"))

# 四个模型真正需要比较什么？

不要把第十章学成“四套安装教程”。

| 维度 | Qwen | ChatGLM3 | Baichuan2 | Yi |
|---|---|---|---|---|
| 教程重点 | Prompt、流式生成、显存 | 生成循环、observation | 特殊 token、截断、量化 | Llama 架构、RoPE、GQA、Streamer |
| 共性 | Causal LM | Causal LM | Causal LM | Causal LM |
| 应长期掌握 | chat template / generate / KV cache | tool loop | context budget / quantization | modern decoder block |

真正迁移到新模型时，先检查：

1. tokenizer / chat template；
2. model class / config；
3. context length；
4. precision / quantization；
5. attention implementation；
6. KV Cache；
7. generation config；
8. tool calling format。

# 一个统一的现代 LLM 推理伪代码

真实项目可以按下面的框架理解，而不是针对每个模型写一套完全不同的逻辑：

```python
messages = [
    {"role": "system", "content": "..."},
    {"role": "user", "content": "..."},
]

inputs = tokenizer.apply_chat_template(
    messages,
    add_generation_prompt=True,
    tokenize=True,
    return_dict=True,
    return_tensors="pt",
).to(model.device)

with torch.inference_mode():
    outputs = model.generate(
        **inputs,
        max_new_tokens=256,
        do_sample=True,
        temperature=0.8,
        top_p=0.9,
        use_cache=True,
    )

new_tokens = outputs[0, inputs["input_ids"].shape[1]:]
answer = tokenizer.decode(new_tokens, skip_special_tokens=True)
```

注意：不同模型的模板、EOS、tool schema、推荐采样参数仍需以模型卡为准。

# 章末知识结构总结

## 1. 模型结构

`CausalLM = Base Decoder Transformer + LM Head`

现代 decoder block 重点：

- causal self-attention
- RoPE
- RMSNorm
- gated MLP
- MHA / GQA / MQA
- residual connection

## 2. 对话层

`messages → chat template → token ids`

多轮对话不是模型内部自动永久保存，而是由应用层管理上下文。

## 3. 生成层

`forward → last logits → sampling → next token → stopping`

流式生成只是边生成边返回。

## 4. 部署层

显存主要关注：

- weights
- KV Cache
- temporary activations / workspace

量化主要降低权重成本；GQA/MQA 还可降低 KV Cache 成本。

## 5. Agent 层

`assistant action → tool → observation → assistant`

工具结果必须回写上下文。

## 6. 应用层

Prompt template、chunking、Map-Reduce、RAG、插件化 workflow，把裸模型变成真正可用的生产力工具。

# 学完必须会回答的 12 个问题

1. 为什么 Qwen、ChatGLM、Baichuan、Yi 都可以抽象成 `tokenizer + CausalLM + generate`？
2. `AutoModelForCausalLM`、base model、decoder blocks、lm_head 分别是什么层级？
3. 多轮对话为什么说“LLM 通常是无状态的”？历史消息在哪里保存？
4. Chat Template 为什么不能随便手写？`add_generation_prompt=True` 的目的是什么？
5. 自回归生成为什么每步只使用最后一个位置的 logits？
6. 流式输出与普通生成在模型计算机制上有什么本质区别？
7. KV Cache 缓存的是什么？为什么它能加速解码，又为什么会占额外显存？
8. MHA、GQA、MQA 对 KV Cache 大小有什么影响？
9. FP16、INT8、INT4 权重显存大致如何估算？为什么真实显存不会完全等于理论值？
10. Context Window 为什么必须同时考虑 prompt tokens 与 `max_new_tokens`？
11. Agent 中 `observation` 为什么不能省略？工具执行失败是否也应该写回 observation？
12. GPT Academic 的文档总结、代码分析和知识库问答分别对应哪些通用 LLM 应用模式？

# 面向后续 LLM / Agent 学习的复习优先级

建议重点顺序：

1. **统一推理流程：messages → template → tokenizer → generate**
2. **KV Cache 与 context length**
3. **现代 Decoder Block：RoPE + RMSNorm + GQA + gated MLP**
4. **量化与显存估算**
5. **tool call → observation 的 Agent 闭环**
6. **RAG：chunk → embedding → retrieval → context**
7. **长文档 Map-Reduce**
8. 四个具体旧模型的安装命令只作为历史工程案例了解